In [1]:
from concurrent.futures import ThreadPoolExecutor

import geopandas as gpd
import numpy as np
import odc.geo  # noqa: F401
from odc.stac import load
from pystac_client import Client
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

from dask.distributed import Client as DaskClient
from utils import predict_xr


In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
fiji_bbox = [177.2, -18.4, 178.9, -17.2]
fiji_bbox_geometry = geometry.box(*fiji_bbox)
fiji_bbox_geometry

# training_file = "training_data/lulc_bare_land.geojson"
training_file = "training_data/draft_inputs/MRD_joined_11.geojson"

tdata = gpd.read_file(training_file, bbox=fiji_bbox_geometry)
tdata.explore()

In [4]:
collection = "dep_ls_geomad"

client = Client.open("https://stac.staging.digitalearthpacific.org")

items = list(client.search(
    collections=[collection],
    bbox=fiji_bbox,
    datetime="2023"
).items())

print(f"Found {len(items)} items")

Found 6 items


In [5]:
bands = [
    "red",
    "green",
    "blue",
    "nir08",
    "swir16",
    "swir22",
    "emad",
    "bcmad",
    "smad",
]

data = load(
    items,
    bbox=fiji_bbox,
    measurements=bands,
    resolution=10,
    chunks={"x": 2000, "y": 2000, "time": 1},
)

data = data.squeeze("time")

data

<xarray.Dataset>
Dimensions:      (y: 13946, x: 18925)
Coordinates:
  * y            (y) float64 -1.931e+06 -1.932e+06 ... -2.071e+06 -2.071e+06
  * x            (x) float64 3.028e+06 3.028e+06 ... 3.217e+06 3.217e+06
    spatial_ref  int32 3832
    time         datetime64[ns] 2023-01-01
Data variables:
    red          (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    green        (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    blue         (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    nir08        (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    swir16       (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    swir22       (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    emad         (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    bcmad        (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    smad         (y, x) float32 dask.array<chunksize=(2000, 2000), meta=np.ndarray>

In [6]:
merged = data.compute()
merged

/opt/homebrew/lib/python3.11/site-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(
/opt/homebrew/lib/python3.11/site-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


<xarray.Dataset>
Dimensions:      (y: 13946, x: 18925)
Coordinates:
  * y            (y) float64 -1.931e+06 -1.932e+06 ... -2.071e+06 -2.071e+06
  * x            (x) float64 3.028e+06 3.028e+06 ... 3.217e+06 3.217e+06
    spatial_ref  int32 3832
    time         datetime64[ns] 2023-01-01
Data variables:
    red          (y, x) float32 7.232e+03 7.224e+03 ... 7.845e+03 7.845e+03
    green        (y, x) float32 7.675e+03 7.678e+03 ... 8.01e+03 8.01e+03
    blue         (y, x) float32 8.032e+03 8.022e+03 ... 7.752e+03 7.752e+03
    nir08        (y, x) float32 7.345e+03 7.346e+03 ... 8.256e+03 8.256e+03
    swir16       (y, x) float32 7.699e+03 7.719e+03 ... 8.776e+03 8.776e+03
    swir22       (y, x) float32 7.692e+03 7.714e+03 ... 8.663e+03 8.663e+03
    emad         (y, x) float32 654.3 701.6 701.6 ... 3.3e+03 3.3e+03 3.3e+03
    bcmad        (y, x) float32 0.01494 0.01599 0.01599 ... 0.07006 0.07006
    smad         (y, x) float32 0.0004953 0.0005348 ... 0.00239 0.00239

In [8]:
from tqdm import tqdm

projected_training_data = tdata.to_crs("EPSG:3832")
training_array = []

def get_training_data(id_row):
    _, row = id_row
    cls_id = row["lulc_code"]
    id = row["id"]
    geom = row["geometry"]

    # Get xarray values at the point
    x = merged.sel(x=geom.x, y=geom.y, method="nearest")
    one_point = [id] + [cls_id] + [float(v) for v in x.values()]
    return one_point

with ThreadPoolExecutor(max_workers=10) as executor:
    training_array = list(tqdm(executor.map(get_training_data, projected_training_data.iterrows()), total=len(projected_training_data)))

print(f"Fetched data for {len(training_array)} training points")

100%|██████████| 753/753 [00:00<00:00, 10734.96it/s]

Fetched data for 753 training points


In [9]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 2:]
classes = np.array(training_array)[:, 1]

model = classifier.fit(training_data, classes)

In [10]:
# This one loads all the data from all Viti Levu
predict_data = merged.fillna(-9999)
predict_data

<xarray.Dataset>
Dimensions:      (y: 13946, x: 18925)
Coordinates:
  * y            (y) float64 -1.931e+06 -1.932e+06 ... -2.071e+06 -2.071e+06
  * x            (x) float64 3.028e+06 3.028e+06 ... 3.217e+06 3.217e+06
    spatial_ref  int32 3832
    time         datetime64[ns] 2023-01-01
Data variables:
    red          (y, x) float32 7.232e+03 7.224e+03 ... 7.845e+03 7.845e+03
    green        (y, x) float32 7.675e+03 7.678e+03 ... 8.01e+03 8.01e+03
    blue         (y, x) float32 8.032e+03 8.022e+03 ... 7.752e+03 7.752e+03
    nir08        (y, x) float32 7.345e+03 7.346e+03 ... 8.256e+03 8.256e+03
    swir16       (y, x) float32 7.699e+03 7.719e+03 ... 8.776e+03 8.776e+03
    swir22       (y, x) float32 7.692e+03 7.714e+03 ... 8.663e+03 8.663e+03
    emad         (y, x) float32 654.3 701.6 701.6 ... 3.3e+03 3.3e+03 3.3e+03
    bcmad        (y, x) float32 0.01494 0.01599 0.01599 ... 0.07006 0.07006
    smad         (y, x) float32 0.0004953 0.0005348 ... 0.00239 0.00239

In [11]:
# This runs the actual prediction
predicted = predict_xr(model, predict_data, proba=True)

predicting...


: 

In [12]:
# Convert to int
cleaned_predictions = predicted.copy(deep=True)
cleaned_predictions.Predictions.data = predicted.Predictions.data.astype(np.int8)
cleaned_predictions.Probabilities.data = predicted.Probabilities.data.astype(np.float32)

cleaned_predictions = cleaned_predictions.rename({"Predictions": "lulc", "Probabilities": "prob"})

cleaned_predictions.prob.plot.imshow(size=10)

NameError: name 'predicted' is not defined

In [13]:
# Write GeoTIFF
cleaned_predictions.prob.odc.write_cog("prob_fiji.tif", overwrite=True)

NameError: name 'cleaned_predictions' is not defined